# Parallel trends
Hola Mariano, por si algún día regresas a este código y ya no te acuerdas de qué estabas haciendo, te dejo aquí el contexto de lo que estás haciendo y por qué: 

1. Tu asesor de tesis te dio comentarios sobre tu tesis, entre los cuales destaca el hecho de que realmente estás haciendo diff-in-diff y no propensity score matching method (solo lo estás usando para construir un grupo de control que realmente sea comparable, pero al final el análisis lo estás haciendo con una regresión lineal de diff-in-diff)
2. Lo que eso implica es que tienes que demostrar que se cumple el supuesto de tendencias paralelas, y como no es en estricto sentido demostrable, tienes que hacer pruebas que al menos no lo nieguen
3. La primera prueba que puedes hacer es la de placebo, que básicamente consiste en tomar una fecha arbitraria previa a la implementación (solo datos antes de la implementación) y correr un diff-in-diff y mostrar que el efecto causal es 0 para el tratamiento.
4. Y la segunda prueba es visuals: cómo se ven las tendencias antes y después de la implementación con y sin propensity score matching.

In [1]:
import os
os.chdir("/Users/mariano/Documents/itam/tesis/speed-cameras/scripts/")

In [2]:
from ps_features_builder import PSFeaturesBuilder
from effect_estimation import EffectEstimation
from ps_matching import PSMatching
import pandas as pd
import pickle

In [3]:
PATH_DATA = "../data/"
PATHS = {
    'vialidades': os.path.join(PATH_DATA, 'vialidades.json'),
    'speed_cameras': os.path.join(PATH_DATA, 'fotocivicas-ubicacion-puntos', 'fotocivicas-ubicacion-puntos.shp'),
    'metro_coordinates': os.path.join(PATH_DATA, 'metro-station-coordinates.parquet'),
    'afluencia_metro': os.path.join(PATH_DATA, 'afluencia-metro-semanal.parquet'),
    'classified_incidents': os.path.join(PATH_DATA, 'classified-incidents.parquet'),
    'volumen_mensual': os.path.join(PATH_DATA, 'volumen-total-mensual.parquet'),
    'ps_feature_builders' : os.path.join(PATH_DATA, "pickle_objects", "ps_features_builder")
}

# grid_type, radio, n_circles
radios = [
    ("circular", 37.5, 25), # 0
    ("circular", 50, 15), # 1
    ("circular", 100, 20), # 2
    ("circular", 150, 15), # 3
    ("circular", 200, 20), # 4
    ("circular", 250, 25), # 5
]

object_name_format = "{grid_type}_{radio}_{n_circles}.pkl"

Para poder hacer el análisis, tengo 3 objetos en python que me ayudan a lograrlo: 
1. Propensity Score Features Builder `PSFeaturesBuilder`: aquí se construye el grid y las unidades que se usan para el análisis, tanto las opciones de control como las que son tratadas. Aquí pensé en una primera instancia volver a crear todo, pero creo que no vale la pena, porque para demostrar tendencias paralelas solo lo voy a hacer con las unidades que estoy usando para mi análisis final, y no volver a encontrar un buen match otra vez.
2. Propensity Score Matching `PSMatching`: Aquí es donde tomo el grid, y hago el match de las unidades tratadas con las de control.

In [30]:
placebo_results = []

In [31]:
grid_type, radio, n_circles = radios[3]
print(radio)

filename = object_name_format.format(
    grid_type=grid_type,
    radio=int(radio),
    n_circles=n_circles
)
obj_path = os.path.join(PATHS.get("ps_feature_builders"), filename)
with open(obj_path, "rb") as f:
    ps_builder : PSFeaturesBuilder = pickle.load(f)

150


In [32]:
# tras haber leído los objetos que ya tienen los grid cargados, hacemos el match
psm = PSMatching(
    ps_features=ps_builder.ps_features,
    grid=ps_builder.grid,
    outcome=ps_builder.outcome,
    grid_type=ps_builder.grid_type,
    grid_size=ps_builder.circle_radius
)
psm.build(
    matching_method="nearest",
    n_matches=1
)

In [36]:
# y una vez que ya se tiene el match, ahora sí hacemos el análisis de los efectos causales
# lo que pasa aquí es que no sé cómo hacerle para poder editar los datos antes de hacer la
# estimación de los efectos. Déjame ver cómo está implementado este pedo.

# Ya ví, nmms va a estar bien fácil. tiene un parámetro para hacer eso. 
# nada más tengo que asegurarme de que le estoy pasando solo datos antes de la implementación real
years = .5
days = int(365*years)
simulated_implementation_date = ps_builder.inicio_operaciones - pd.DateOffset(days=days)

effect_estimator = EffectEstimation(
    outcome=psm.outcome[psm.outcome.timestamp < ps_builder.inicio_operaciones],
    matched_grids=psm.matched_grids,
    inicio_operaciones=simulated_implementation_date
)
effect_estimator.estimate_all()

(
    effect_estimator
    .get_all_treatment_effects()
    .assign(radius_size=psm.grid_size, years_offset=years)
    .pipe(lambda df: placebo_results.append(df))
)

In [37]:
# ahora ponemos todos en una única tabla
pd.concat(placebo_results, ignore_index=True).sort_values(["radius_size", "years_offset"])

,outcome_type,variable,coeficiente,error_estandar,valor_p,valor_p_fe,radius_size,years_offset
16,total,total,-5.107527e-02,1.832334e-01,0.780441,0.689799,150,0.5
17,total,min,-4.238161e-02,1.258889e-01,0.736374,0.661545,150,0.5
18,total,pic,-1.069549e-02,9.670817e-02,0.911937,0.884534,150,0.5
19,total,fcs,2.001830e-03,8.723634e-03,0.818502,0.810086,150,0.5
20,tasas,total,-3.449002e-06,1.148841e-05,0.764013,0.666750,150,0.5
21,tasas,min,-2.798110e-06,7.877126e-06,0.722425,0.643824,150,0.5
22,tasas,pic,-7.902025e-07,6.090310e-06,0.896766,0.864945,150,0.5
23,tasas,fcs,1.393107e-07,5.524655e-07,0.800916,0.792452,150,0.5
8,total,total,7.180851e-02,1.451187e-01,0.620723,0.562534,150,1.0
9,total,min,4.978723e-02,1.011151e-01,0.622449,0.574880,150,1.0
